# A7 — split direct + indirect, russian roulette

Sample the light explicitly for direct term; bounce for indirect, terminate with RR.

Have to be careful not to double-count: the recursive indirect path skips emitters.

In [1]:
import numpy as np

P_RR = 0.8

def cast_ray(scene, ray, depth):
    hit = scene.intersect(ray)
    if hit is None:
        return np.zeros(3)
    if hit.material.is_emissive:
        # only counted directly by primary rays; indirect skips this
        return hit.material.emit if depth == 0 else np.zeros(3)

    L_dir = np.zeros(3)
    light_hit, pdf_light = scene.sample_light()
    to_light = light_hit.pos - hit.pos
    dist2 = float(to_light @ to_light)
    wi = to_light / np.sqrt(dist2)
    # shadow ray
    occl = scene.intersect(Ray(hit.pos, wi))
    if occl is not None and abs(occl.t * occl.t - dist2) < 1e-2:
        brdf = hit.material.albedo / np.pi
        cos1 = max(0.0, hit.normal @ wi)
        cos2 = max(0.0, light_hit.normal @ (-wi))
        L_dir = light_hit.material.emit * brdf * cos1 * cos2 / dist2 / pdf_light

    L_ind = np.zeros(3)
    if np.random.random() < P_RR:
        wo = sample_hemisphere(hit.normal)
        pdf = 1 / (2 * np.pi)
        Li = cast_ray(scene, Ray(hit.pos, wo), depth + 1)
        brdf = hit.material.albedo / np.pi
        cos_t = max(0.0, hit.normal @ wo)
        L_ind = Li * brdf * cos_t / pdf / P_RR

    return L_dir + L_ind
